# RAG With llama-index  + Milvus + LLama

References
- https://docs.llamaindex.ai/en/stable/examples/vector_stores/MilvusIndexDemo/
- https://docs.llamaindex.ai/en/stable/api_reference/storage/vector_store/milvus/?h=milvusvectorstore#llama_index.vector_stores.milvus.MilvusVectorStore

## Step-1: Configuration

In [3]:
from my_config import MY_CONFIG

MY_CONFIG.DB_URI = './rag_2_llamaindex.db'
MY_CONFIG.COLLECTION_NAME = 'llamaindex_papers'

## Step-2: Setup Embeddings

In [4]:
# If connection to https://huggingface.co/ failed, uncomment the following path
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

In [5]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

Settings.embed_model = HuggingFaceEmbedding(
    model_name = MY_CONFIG.EMBEDDING_MODEL
)

## Step-3: Connect to Milvus

In [6]:
# connect to vector db
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.milvus import MilvusVectorStore

vector_store = MilvusVectorStore(
    uri = MY_CONFIG.DB_URI ,
    dim = MY_CONFIG.EMBEDDING_LENGTH , 
    collection_name = MY_CONFIG.COLLECTION_NAME,
    overwrite=False  # so we load the index from db
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print ("✅ Connected Llama-index to Milvus instance: ", MY_CONFIG.DB_URI )

✅ Connected Llama-index to Milvus instance:  ./rag_2_llamaindex.db


## Step-4: Load Document Index from DB

In [7]:
%%time

from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, storage_context=storage_context)

print ("✅ Loaded index from vector db:", MY_CONFIG.DB_URI )

✅ Loaded index from vector db: ./rag_2_llamaindex.db
CPU times: user 95 ms, sys: 10.4 ms, total: 105 ms
Wall time: 104 ms


## Step-5: Setup LLM

In [8]:
from llama_index.llms.replicate import Replicate
from llama_index.core import Settings

llm = Replicate(
    model= MY_CONFIG.LLM_MODEL,
    temperature=0.1
)

Settings.llm = llm

## Step-6: Query

In [9]:
query_engine = index.as_query_engine()
res = query_engine.query("What was the training data used to train Granite models?")
print(res)

The Granite models were trained using a combination of code data and natural language datasets. The code data was collected for model training, while the natural language datasets included web documents (Stackexchange, CommonCrawl), mathematical web text (OpenWebMath, StackMathQA), academic text (Arxiv, Wikipedia), and instruction tuning datasets (FLAN, HelpSteer). These datasets were not deduplicated. Additionally, the datasets were scanned using ClamAV to identify and remove instances of malware in the source code. Personal Identifiable Information (PII) such as keys, email addresses, names, user names, and passwords were redacted and replaced with corresponding tokens. IP addresses were also changed with synthetically generated IP addresses.


In [10]:
query_engine = index.as_query_engine()
res = query_engine.query("What is attention mechanism?")
print(res)

The attention mechanism is a concept used in machine learning, particularly in models like transformers. It allows the model to focus on different parts of the input data when producing an output, mimicking the cognitive process of attention in humans.

In the context of natural language processing, the attention mechanism helps the model understand the context of words in a sentence by weighing their importance relative to each other. This is done by computing a set of attention scores, which determine the weight or importance of each word in the input sequence when generating the output.

The attention mechanism can follow long-distance dependencies, as demonstrated in the example provided. In this case, many attention heads in layer 5 of the encoder self-attention attend to a distant dependency of the verb 'making', completing the phrase 'making...more difficult'.

There are different types of attention mechanisms, such as Scaled Dot-Product Attention and Multi-Head Attention. Scale

In [11]:
query_engine = index.as_query_engine()
res = query_engine.query("When was the moon landing?")
print(res)

The provided context does not contain information about the moon landing. Therefore, I cannot provide an answer to your query. The text discusses IBM Granite Code Models, a family of open foundation models for code intelligence.
